In [4]:
import pandas as pd
import numpy as np

Hierarchical indexing enables you to have multiple index on an axis.
```python
data = pd.Series(np.random.uniform(size=9), index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"], [1, 2, 3, 1, 3, 1, 2, 2, 3]])
```

You can select subsets using *partial indexing*

In [9]:
data
#data["b"]

a  1    0.511641
   2    0.813593
   3    0.841838
b  1    0.346413
   3    0.482314
c  1    0.340859
   2    0.404638
d  2    0.062716
   3    0.587352
dtype: float64

In [13]:
data["b":"c"]

b  1    0.346413
   3    0.482314
c  1    0.340859
   2    0.404638
dtype: float64

In [14]:
data.loc[["b", "d"]]

b  1    0.346413
   3    0.482314
d  2    0.062716
   3    0.587352
dtype: float64

Selection is even possible from an “inner” level

In [19]:
data[:,2]

a    0.813593
c    0.404638
d    0.062716
dtype: float64

Rearrange this data into a DataFrame using its `unstack` method. And its inverse operation `stack`

In [20]:
data.unstack()

,1,2,3
a,0.511641,0.813593,0.841838
b,0.346413,NaN,0.482314
c,0.340859,0.404638,NaN
d,NaN,0.062716,0.587352


With a DataFrame, either axis can have a hierarchical index

In [31]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)), index=[["a", "a", "b", "b"], [1, 2, 1, 2]], columns=[["Ohio", "Ohio", "Colorado"],["Green", "Red", "Green"]])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

The hierarchical levels can have names (as strings or any Python objects)

In [39]:
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

You can see how many levels an index has by accessing its nlevels attribute
```python
frame.index.nlevels
```

With partial column indexing you can similarly select groups of columns.
```python
frame["Ohio"]
```
A MultiIndex can be created by itself and then reused; the columns in the preceding
DataFrame with level names could also be created.
```python
pd.MultiIndex.from_arrays([["Ohio", "Ohio", "Colorado"],
 ["Green", "Red", "Green"]],
 names=["state", "color"])
```

### Reordering and Sorting Levels

The swaplevel method takes two level numbers
or names and returns a new object with the levels interchanged

In [40]:
frame.swaplevel("key1", "key2")

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

sort_index by default sorts the data lexicographically using all the index levels, but
you can choose to use only a single level or a subset of levels to sort by passing the
level argument.

In [45]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

### Summary Statistics by Level

In [50]:
frame.groupby(level="key2").sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [53]:
frame.groupby(level="color", axis="columns").sum()

/tmp/nix-shell-18190-3474373290/ipykernel_19733/775557097.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  frame.groupby(level="color", axis="columns").sum()


color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

### Indexing with a DataFrame’s columns

It’s not unusual to want to use one or more columns from a DataFrame as the
row index; alternatively, you may wish to move the row index into the DataFrame’s
columns. 

In [56]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
"c": ["one", "one", "one", "two", "two",
"two", "two"], "d": [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame’s *set_index* function will create a new DataFrame using one or more of
its columns as the index.*reset_index*, on the other hand, does the opposite of set_index; the hierarchical
index levels are moved into the columns

In [58]:
frame2 = frame.set_index(["c", "d"])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [60]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1
